# Day 10 · Read the failures, not the averages

**First-principles question:** Why does it fail, and which fix matters most?

Today is one turn of an OODA loop: **Observe** the failures, **Orient** them against the day 3 failure modes,
**Decide** on one fix, **Act** by re-running and scoring again.

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import csv
import pandas as pd
import matplotlib.pyplot as plt
from lab.acme import load_cases, load_results
from lab.evals import score

cases = {c["id"]: c for c in load_cases()}
FAILURES = Path("failures.csv")    # in this folder

## 1 · Observe: collect every failure into a spreadsheet

In [ ]:
rows = []
for version in ["A", "B", "C"]:
    for r in load_results(version):
        s = score(cases[r["id"]], r)
        if not s["passed"]:
            failed = [k for k in ("intent_ok", "facts_ok", "policy_ok", "human_ok") if not s[k]]
            rows.append({"key": f"{r['id']}:{version}", "case": r["id"], "version": version,
                         "failed_checks": " ".join(failed), "violations": " ".join(s["violations"]),
                         "note": "", "category": ""})

if FAILURES.exists():   # keep the notes you already wrote
    old = {r["key"]: r for r in csv.DictReader(FAILURES.open(encoding="utf-8"))}
    for r in rows:
        r["note"], r["category"] = old.get(r["key"], {}).get("note", ""), old.get(r["key"], {}).get("category", "")
with FAILURES.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
print(len(rows), "failures written to", FAILURES.resolve())

## 2 · Observe: read one failure at a time
**TODO:** read at least 20 failures, spread across A, B and C. For each one, write a one-line **note** in
`failures.csv` (VS Code or Numbers) saying what went wrong in your own words. Leave **category** empty for now.
Re-run step 1 any time: your notes are kept.

In [ ]:
def show(key):
    case_id, version = key.split(":")
    case = cases[case_id]
    out = next(r for r in load_results(version) if r["id"] == case_id)
    print(f"── {key} · expected {case['expected_intent']} · must include {case['must_include']} · "
          f"needs_human {case['needs_human']}")
    print("EMAIL:", case["message"])
    print(f"OUTPUT: intent={out.get('intent')} needs_human={out.get('needs_human')}\n{out.get('reply')}")
    print("CHECKS:", score(case, out))

show(rows[0]["key"])

## 3 · Orient: fit each note to a failure mode
**TODO:** fill in the **category** column. Start from your day 3 modes (unsupported-amount false positive,
over-escalation on an answerable case, unapproved offer). Add a new category (wrong tool, gave up, bad JSON…)
only when no existing one fits the note.

In [ ]:
notes = pd.read_csv(FAILURES).fillna("")
labelled = notes[(notes.note != "") & (notes.category != "")]
pivot = pd.crosstab(labelled.category, labelled.version, margins=True)
pivot.drop("All").drop(columns="All").plot.barh(stacked=True, figsize=(7, 3), title="Failures by category")
plt.show()

# Where the architectures differ: categories seen in exactly one version
by_version = pd.crosstab(labelled.category, labelled.version)
only_one = by_version[(by_version > 0).sum(axis=1) == 1]
for category in only_one.index:
    example = labelled[labelled.category == category].iloc[0]
    print(f"only in {example.version}: {category} (e.g. {example.case})")
pivot

## 4 · Decide: one fix, with a prediction
**TODO:** pick B or C (whichever you would ship after day 9) and the single fix that should remove the most
failures. Write the prediction in the next cell **before** you change anything, so the result can prove you wrong.

In [ ]:
# TODO: your decision
FIX_VERSION = ""          # "B" or "C"
FIX_CATEGORY = ""         # the category from step 3 that the fix targets
PREDICTED_REMOVED = 0     # how many of that version's failures you expect it to remove
FIX = ""                  # the one instruction line you will add to that version's system prompt

## 5 · Act: apply the fix, re-run, score again
The next cell adds `FIX` to the version's prompt, re-runs all 84 cases with the same model setup as day 9,
and saves the outputs as `B2` or `C2`. If your fix is a code change in `lab/` instead, leave `FIX` empty:
the cell reloads `lab/` first.

Compare every case, not only the failures: a fix that breaks passing cases is a regression.
Afterwards, `show("c12:B2")` opens a re-run case in the viewer. That is your next Observe.

In [ ]:
import importlib
import lab.agent, lab.version_b
from lab.acme import save_results

assert FIX_VERSION in ("B", "C"), "Fill in your decision in step 4 first"

# Reload so re-running this cell never stacks the fix twice
vb = importlib.reload(lab.version_b)
ag = importlib.reload(lab.agent)
if FIX and FIX_VERSION == "B":
    vb.SYSTEM += "\n" + FIX
elif FIX:
    ag.SYSTEM_C += "\n" + FIX

def run(case):
    if FIX_VERSION == "B":
        return vb.version_b(case)
    output, trace = ag.run_agent(case)
    return {**output, "steps": len(trace)}

new_rows = [{**run(case), "id": cid} for cid, case in cases.items()]
save_results(f"{FIX_VERSION}2", new_rows)

before = {r["id"]: score(cases[r["id"]], r)["passed"] for r in load_results(FIX_VERSION)}
after = {r["id"]: score(cases[r["id"]], r)["passed"] for r in new_rows}
fixed = [c for c in cases if after[c] and not before[c]]
broke = [c for c in cases if before[c] and not after[c]]
print(f"{FIX_VERSION} → {FIX_VERSION}2: {sum(before.values())} → {sum(after.values())} passed of {len(cases)}")
print(f"Fixed {len(fixed)} (you predicted {PREDICTED_REMOVED}):", fixed)
print(f"Broke {len(broke)}:", broke)

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(labelled) >= 20, f"Label at least 20 failures (note + category): {len(labelled)} so far"
missing = {"A", "B", "C"} - set(labelled.version)
assert not missing, f"Label failures from every version (none yet from {', '.join(sorted(missing))})"
assert len(only_one) >= 1, "Find at least one category that appears in only one version"
assert FIX_CATEGORY in set(labelled.category), "FIX_CATEGORY must be one of your categories from step 3"
assert PREDICTED_REMOVED > 0, "Write your prediction in step 4 before acting"
assert "after" in globals(), "Run step 5 (Act) to re-score the fixed version"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")